# CallGuard AI - Notebook 04: Recruitment Call Detection & Legitimacy Classification

### Objective
Build a hierarchical two-stage classification pipeline for employment telephony interactions:
1. **Stage 1: Recruitment vs Non-Recruitment**: Identify calls originating from recruiters, talent acquisition, or interview schedulers.
2. **Stage 2: Legitimate vs Fraudulent Recruitment**: Protect job seekers by distinguishing genuine career opportunities from advance-fee recruitment scams (equipment fee demands, wire transfers, fake interviews).

In [ ]:
# Cell 2: Install dependencies & import libraries
!pip install -q scikit-learn pandas matplotlib seaborn joblib

import os
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, f1_score

print("Recruitment modeling dependencies loaded.")

In [ ]:
# Cell 3: Load CallGuard custom dataset
data_path = Path("ml/datasets/callguard/processed/cleaned_full.jsonl")
if not data_path.exists():
    data_path = Path("ml/datasets/callguard/synthetic_conversations.jsonl")

with open(data_path, "r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

df = pd.DataFrame(records)
print(f"Loaded {len(df)} calls from {data_path}")

# Display distribution of recruitment scenarios
print("\nBreakdown of recruitment vs other calls:")
print(df["is_recruitment"].value_counts())

In [ ]:
# Cell 4: Binary classification: recruitment vs non-recruitment
text_col = "full_transcript"
X = df[text_col]
y_stage1 = df["is_recruitment"].astype(int)

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y_stage1, df.index, test_size=0.2, random_state=42, stratify=y_stage1
)

tfidf_stage1 = TfidfVectorizer(max_features=4000, ngram_range=(1, 2), stop_words="english")
X_train_vec = tfidf_stage1.fit_transform(X_train)
X_test_vec = tfidf_stage1.transform(X_test)

print(f"Stage 1 Train samples: {X_train_vec.shape[0]}, Test samples: {X_test_vec.shape[0]}")

In [ ]:
# Cell 5: TF-IDF + LR for recruitment detection
# We configure class_weight to ensure high recall for legitimate job opportunities
lr_stage1 = LogisticRegression(class_weight="balanced", random_state=42)
lr_stage1.fit(X_train_vec, y_train)
print("Stage 1 recruitment detector trained.")

In [ ]:
# Cell 6: Evaluate recruitment detector
y_pred_s1 = lr_stage1.predict(X_test_vec)

print("=== Stage 1: Recruitment vs Non-Recruitment Evaluation ===")
print(classification_report(y_test, y_pred_s1, target_names=["Non-Recruitment", "Recruitment"]))

# Emphasize recall metric
from sklearn.metrics import recall_score, precision_score
rec = recall_score(y_test, y_pred_s1)
prec = precision_score(y_test, y_pred_s1)
print(f"Recruitment Recall (Safety against missing career calls): {rec:.4f}")
print(f"Recruitment Precision: {prec:.4f}")

In [ ]:
# Cell 7: Recruitment legitimacy classifier (Stage 2)
# Filter dataset to only recruitment calls
recruitment_df = df[df["is_recruitment"] == True].copy().reset_index(drop=True)
print(f"Stage 2 Sub-dataset: {len(recruitment_df)} recruitment conversations.")
print(recruitment_df["is_legitimate"].value_counts())

X_s2 = recruitment_df[text_col]
y_s2 = recruitment_df["is_legitimate"].astype(int)

X_s2_train, X_s2_test, y_s2_train, y_s2_test = train_test_split(
    X_s2, y_s2, test_size=0.25, random_state=42, stratify=y_s2
)

tfidf_stage2 = TfidfVectorizer(max_features=3000, ngram_range=(1, 2), stop_words="english")
X_s2_train_vec = tfidf_stage2.fit_transform(X_s2_train)
X_s2_test_vec = tfidf_stage2.transform(X_s2_test)

lr_stage2 = LogisticRegression(random_state=42)
lr_stage2.fit(X_s2_train_vec, y_s2_train)

y_pred_s2 = lr_stage2.predict(X_s2_test_vec)
print("\n=== Stage 2: Recruitment Legitimacy Evaluation ===")
print(classification_report(y_s2_test, y_pred_s2, target_names=["Fraudulent Job", "Legitimate Job"]))

In [ ]:
# Cell 8: Feature importance analysis
feature_names = np.array(tfidf_stage2.get_feature_names_out())
coefs = lr_stage2.coef_[0]

top_legit_idx = np.argsort(coefs)[-10:]
top_fraud_idx = np.argsort(coefs)[:10]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].barh(feature_names[top_fraud_idx], np.abs(coefs[top_fraud_idx]), color="crimson")
axes[0].set_title("Top Words Indicating Job Scams / Advance Fee Fraud", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Feature Weight (Abs)")

axes[1].barh(feature_names[top_legit_idx], coefs[top_legit_idx], color="forestgreen")
axes[1].set_title("Top Words Indicating Legitimate Recruitment", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Feature Weight")

plt.tight_layout()
plt.show()

In [ ]:
# Cell 9: Save models
model_dir = Path("ml/models")
model_dir.mkdir(parents=True, exist_ok=True)

s1_artifact = {
    "vectorizer": tfidf_stage1,
    "model": lr_stage1,
    "description": "Stage 1: Recruitment call detector"
}
s2_artifact = {
    "vectorizer": tfidf_stage2,
    "model": lr_stage2,
    "description": "Stage 2: Recruitment legitimacy classifier"
}

joblib.dump(s1_artifact, model_dir / "recruitment_detector_v1.0.0.joblib")
joblib.dump(s2_artifact, model_dir / "recruitment_legitimacy_v1.0.0.joblib")

meta = {
    "pipeline": "hierarchical_recruitment_classifier",
    "version": "1.0.0",
    "stage1_model": "recruitment_detector_v1.0.0.joblib",
    "stage2_model": "recruitment_legitimacy_v1.0.0.joblib",
    "stage1_recall": float(rec),
    "stage2_f1": float(f1_score(y_s2_test, y_pred_s2))
}

with open(model_dir / "recruitment_pipeline_v1.0.0.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

print("Saved recruitment models and pipeline metadata to ml/models/")

# Cell 10: Key findings & limitations

### Findings:
1. **Clear Fraud Indicators**: Phrases such as *equipment fee*, *zelle*, *crypto*, *refundable onboarding*, and *no interview required* strongly signal advance fee recruitment fraud.
2. **Legitimate Patterns**: Real recruiters reference specific resumes, LinkedIn profiles, scheduling slots, and technical director interviews.
3. **Operational Thresholding**: Stage 1 prioritizes high recall (>0.98) so genuine employers are never blocked, while Stage 2 flags suspicious advance-fee requests for immediate human review.